In [42]:
import pandas as pd
import os 
import tarfile
import zstandard as zstd
import json

def of3_json(chain_dict, model_name, num_recycles=3, json_path="of3_input.json", use_paired_msas=False):
    """
    Makes an input json file for OF3 given a chain_dict.

    Args:
        chain_dict (dict): A dictionary where keys are chain IDs and values are dictionaries with keys
            "seq" (str): The sequence of the chain.
            "msa_path" (str): The path to the MSA file for the chain.
        model_name (str): The name of the model to use in OF3.
        num_recycles (int, optional): The number of recycles to use in OF3. Defaults to 3.
        json_path (str, optional): The path to save the output json file. Defaults to "of3_input.json".
        use_paired_msas (bool, optional): Whether to use paired MSAs in OF3. Defaults to False.

    Returns:
        None: The function saves the input json file to the specified path.
    """

    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    # Required keys
    json_dict = {"queries": {model_name: {}}}
    for chain in chain_dict.keys():
        if "chains" not in json_dict["queries"][model_name]:
            json_dict["queries"][model_name]["chains"] = []

        json_dict["queries"][model_name]["chains"].append({
            "molecule_type": "protein",
            "chain_ids": [chain],
            "sequence": chain_dict[chain]["seq"],
            "main_msa_file_paths": [chain_dict[chain]["msa_path"]],
        })
    
    json_dict["queries"][model_name]["model"] = {
        "num_recycles": num_recycles
    }
    if not use_paired_msas:
        json_dict["queries"][model_name]["use_paired_msas"] = use_paired_msas

    # Dump to json
    with open(json_path, 'w') as f:
        json.dump(json_dict, f, indent=4)

def extract_files(file_path, target_suffix="_data.json", output_dir="extracted_data"):
    """
    Extracts specific files given a suffix from a .tar.zst archive.

    Args:
        file_path (str): The path to the .tar.zst archive.
        target_suffix (str, optional): The suffix of the files to extract. Defaults to "_data.json".
        output_dir (str, optional): The directory to save the extracted files. Defaults to "extracted_data".

    Returns:
        None: The function extracts the files to the specified output directory.
    """
    # Ensure output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Open the .tar.zst file and extract files with the target suffix
    with open(file_path, 'rb') as f:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(f) as reader:
            with tarfile.open(fileobj=reader, mode='r|') as tar:
                for member in tar:
                    if member.isfile() and member.name.endswith(target_suffix):
                        filename = os.path.basename(member.name)
                        dest_path = os.path.join(output_dir, filename)
                        with open(dest_path, 'wb') as f_out:
                            f_out.write(tar.extractfile(member).read())


def fasta_to_dict(fasta_file):
    """
    Reads a FASTA file and returns a dictionary where keys are headers (without '>') and values are sequences.

    Args:   
        fasta_file (str): The path to the FASTA file.

    Returns:
        dict: A dictionary with headers as keys and sequences as values.
    """
    fasta_dict = {}
    with open(fasta_file, 'r') as f:
        current_header = None
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                current_header = line[1:]
                fasta_dict[current_header] = ""
            elif current_header is not None:
                fasta_dict[current_header] += line
    return fasta_dict

# Get directory of the script
script_dir = os.path.dirname(os.getcwd())
repo_name = "WP2_SweetProteinDiscovery"
repo_dir = script_dir.split(repo_name)[0] + repo_name
cluster_dir = f"{repo_dir}/6_structural_analyses/2_mmseqs/clustered_hits"
cluster_dirs = [os.path.join(cluster_dir, d) for d in os.listdir(cluster_dir) if os.path.isdir(os.path.join(cluster_dir, d))]
cluster_files = []
for d in cluster_dirs:
    for f in os.listdir(d):
        if f.endswith("_cluster.tsv"):
            cluster_files.append(os.path.join(d, f))

# Read tm_score results for OneKP and UniProt
filtered_tm_scores_df = pd.read_csv(f"{repo_dir}/6_structural_analyses/1_tm_scores/tm_scores_filtered.tsv", sep="\t")
for index, row in filtered_tm_scores_df.iterrows():
    if row["source"] == "UniProt":
        filtered_tm_scores_df.at[index, 'pdb'] = row["query"] + "___" + row['target'].split('|')[1]
    elif row["source"] == "OneKP":
        filtered_tm_scores_df.at[index, 'pdb'] = row["query"] + "___" + row['target']

# Remove high identity matches (pident >= 90%) to avoid redundancy
print(f"Number of rows before removing high identity matches: {len(filtered_tm_scores_df)}")
filtered_tm_scores_df = filtered_tm_scores_df[filtered_tm_scores_df['pident'] < 90]
print(f"Number of rows after removing high identity matches: {len(filtered_tm_scores_df)}")

# Read cropped fasta sequences
fasta_dir = f"{repo_dir}/6_structural_analyses/1_tm_scores/merged_hits"
fasta_dfs = []
for sweet_protein in filtered_tm_scores_df["query"].unique():
    fasta_file = os.path.join(fasta_dir, f"{sweet_protein}_merged_hits.fasta")
    if os.path.exists(fasta_file):
        fasta_dict = fasta_to_dict(fasta_file)
        # Make into a df 
        fasta_df = pd.DataFrame(list(fasta_dict.items()), columns=["pdb", "cropped_tseq"])
        fasta_dfs.append(fasta_df)
fasta_df = pd.concat(fasta_dfs, ignore_index=True)

# Merge tm_score results with fasta sequences
filtered_tm_scores_df = pd.merge(filtered_tm_scores_df, fasta_df, on="pdb", how="left")

Number of rows before removing high identity matches: 10015
Number of rows after removing high identity matches: 10008


In [43]:
# Load Thaumatin I and II cluster information
thaumatin_clusters = []
for cluster_file in cluster_files:
    df = pd.read_csv(cluster_file, sep="\t", header=None, names=["cluster_id", "pdb"])
    thaumatin_clusters.append(df)

# Concatenate all cluster dataframes into one
thaumatin_cluster_df = pd.concat(thaumatin_clusters, ignore_index=True)
thaumatin_cluster_df = thaumatin_cluster_df.drop_duplicates()

# Merge tm_score results with cluster information
thaumatin_df = pd.merge(filtered_tm_scores_df, thaumatin_cluster_df, on="pdb", how="inner")

# Get best TM-score for each cluster 
thaumatin_df = thaumatin_df.loc[thaumatin_df.groupby("cluster_id")["grantham_score"].idxmin()]

# From filtered_tm_scores_df, drop Thaumatin hits
filtered_tm_scores_df = filtered_tm_scores_df[~filtered_tm_scores_df['query'].str.contains("Thaumatin")]

# Concatenate thaumatin_df with the rest of the filtered_tm_scores_df
representative_hits_df = pd.concat([filtered_tm_scores_df, thaumatin_df], ignore_index=True)

# Write out fastas for the representative hits
fasta_dir = f"{repo_dir}/6_structural_analyses/3_representative_structures/fastas/peptides"
os.makedirs(fasta_dir, exist_ok=True)
for index, row in representative_hits_df.iterrows():
    fasta_file = os.path.join(fasta_dir, f"{row['pdb']}.fasta")
    with open(fasta_file, 'w') as f:
        f.write(f">{row['pdb']}\n{row['cropped_tseq']}\n")

# Copy sweet protein fastas to the same folder
sweet_protein_fastas = os.listdir(f"{repo_dir}/1_viridiplantae_prot_db/4_sweet_proteins/sweet_protein_fastas/")
for fasta in sweet_protein_fastas:
    src_path = f"{repo_dir}/1_viridiplantae_prot_db/4_sweet_proteins/sweet_protein_fastas/{fasta}"
    dst_path = os.path.join(fasta_dir, fasta)
    if not os.path.exists(dst_path):
        os.system(f"cp {src_path} {dst_path}")

receptor_seq_dict = {
    "ts1r2_human": "EPAENSDFYLPGDYLLGGLFSLHANMKGIVHLNFLQVPMCKEYEVKVIGYNLMQAMRFAVEEINNDSSLLPGVLLGYEIVDVCYISNNVQPVLYFLAHEDNLLPIQEDYSNYISRVVAVIGPDNSESVMTVANFLSLFLLPQITYSAISDELRDKVRFPALLRTTPSADHHIEAMVQLMLHFRWNWIIVLVSSDTYGRDNGQLLGERVARRDICIAFQETLPTLQPNQNMTSEERQRLVTIVDKLQQSTARVVVVFSPDLTLYHFFNEVLRQNFTGAVWIASESWAIDPVLHNLTELRHLGTFLGITIQSVPIPGFSEFREWGPQAGPPPLSRTSQSYTCNQECDNCLNATLSFNTILRLSGERVVYSVYSAVYAVAHALHSLLGCDKSTCTKRVVYPWQLLEEIWKVNFTLLDHQIFFDPQGDVALHLEIVQWQWDRSQNPFQSVASYYPLQRQLKNIQDISWHTINNTIPMSMCSKRCQSGQKKKPVGIHVCCFECIDCLPGTFLNHTEDEYECQACPNNEWSYQSETSCFKRQLVFLEWHEAPTIAVALLAALGFLSTLAILVIFWRHFQTPIVRSAGGPMCFLMLTLLLVAYMVVPVYVGPPKVSTCLCRQALFPLCFTICISCIAVRSFQIVCAFKMASRFPRAYSYWVRYQGPYVSMAFITVLKMVIVVIGMLATGLSPTTRTDPDDPKITIVSCNPNYRNSLLFNTSLDLLLSVVGFSFAYMGKELPTNYNEAKFITLSMTFYFTSSVSLCTFMSAYSGVLVTIVDLLVTVLNLLAISLGYFGPKCYMILFYPERNTPAYFNSMIQGYTMRRD",
    "ts1r3_human": "APLCLSQQLRMKGDYVLGGLFPLGEAEEAGLRSRTRPSSPVCTRFSSNGLLWALAMKMAVEEINNKSDLLPGLRLGYDLFDTCSEPVVAMKPSLMFLAKAGSRDIAAYCNYTQYQPRVLAVIGPHSSELAMVTGKFFSFFLMPQVSYGASMELLSARETFPSFFRTVPSDRVQLTAAAELLQEFGWNWVAALGSDDEYGRQGLSIFSALAAARGICIAHEGLVPLPRADDSRLGKVQDVLHQVNQSSVQVVLLFASVHAAHALFNYSISSRLSPKVWVASEAWLTSDLVMGLPGMAQMGTVLGFLQRGAQLHEFPQYVKTHLALATDPAFCSALGEREQGLEEDVVGQRCPQCDCITLQNVSAGLNHHQTFSVYAAVYSVAQALHNTLQCNASGCPAQDPVKPWQLLENMYNLTFHVGGLPLRFDSSGNVDMEYDLKLWVWQGSVPRLHDVGRFNGSLRTERLKIRWHTSDNQKPVSRCSRQCQEGQVRRVKGFHSCCYDCVDCEAGSYRQNPDDIACTFCGQDEWSPERSTRCFRRRSRFLAWGEPAVLLLLLLLSLALGLVLAALGLFVHHRDSPLVQASGGPLACFGLVCLGLVCLSVLLFPGQPSPARCLAQQPLSHLPLTGCLSTLFLQAAEIFVESELPLSWADRLSGCLRGPWAWLVVLLAMLVEVALCTWYLVAFPPEVVTDWHMLPTEALVHCRTRSWVSFGLAHATNATLAFLCFLGTFLVRSQPGCYNRARGLTFAMLAYFITWVSFVPLLANVQVVLRPAVQMGALLLCVLGILAAFHLPRCYLLMRQPGLNTPEFFLGGGPGDAQGQNDGNTGNQGKHE"
}

# Save receptor sequences to fasta files
receptor_fasta_dir = f"{repo_dir}/6_structural_analyses/3_representative_structures/fastas/receptors"
os.makedirs(receptor_fasta_dir, exist_ok=True)
for receptor, seq in receptor_seq_dict.items():
    fasta_file = os.path.join(receptor_fasta_dir, f"{receptor}.fasta")
    with open(fasta_file, 'w') as f:
        f.write(f">{receptor}\n{seq}\n")

# Copy chosen structures for each sweet protein to a new folder

In [45]:
# Make a dictionary to save the files for the best hits for each cluster
rep_structures_path = f"{repo_dir}/6_structural_analyses/3_representative_structures/"
if not os.path.exists(rep_structures_path):
    os.makedirs(rep_structures_path)

# Save resulting dataframe in the rep_structures_path
representative_hits_df.to_csv(os.path.join(rep_structures_path, "representative_structures.csv"), index=False)

# structure path
structure_path = f"{repo_dir}/6_structural_analyses/1_tm_scores/cropped_structures"

# Save these structures 
for index, row in representative_hits_df.iterrows():
    if row["pident"] == 100:
        print(f"Skipping {row['pdb']} with 100% sequence identity")
        continue

    if row["model"] == "AFDB":
        pdb_filename = f"cropped_afdb_{row['pdb']}.pdb"
    elif row["model"] == "AF3":
        pdb_filename = f"cropped_af3_{row['pdb']}.pdb"
    elif row["model"] == "OF3":
        pdb_filename = f"cropped_of3_{row['pdb']}.pdb"

    dst_path = os.path.join(rep_structures_path, row["query"], "pdb_dir", pdb_filename)
    src_path = os.path.join(structure_path, row["source"], pdb_filename)
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    if os.path.exists(src_path):
        if not os.path.exists(dst_path):
            os.system(f"cp {src_path} {dst_path}")
    else:
        print(f"File {src_path} does not exist. Skipping copy.")

# Get highest TM-score hit for each sweet protein and make Pymol visualization scripts

In [46]:
# Get highest TM-score for each query and print the best hit for each query
best_hits_df = representative_hits_df.loc[representative_hits_df.groupby("query")["tm_score"].idxmax()]   
rep_structures_path = f"{repo_dir}/6_structural_analyses/3_representative_structures/"
pymol_script_path = f"{repo_dir}/6_structural_analyses/3_representative_structures/pymol_scripts"
os.makedirs(pymol_script_path, exist_ok=True)

onekp_hits_dict = {}
pymol_script = "set grid_mode, 1\n"
c = 1
for index, row in best_hits_df.iterrows():
    sweet_protein = row["query"]
    pdb_path = os.path.join(rep_structures_path, sweet_protein, "pdb_dir", f"cropped_{row['model'].lower()}_{row['pdb']}.pdb")
    pdb_name = f"cropped_{row['model'].lower()}_{row['pdb']}"
    print(f"Best hit for {sweet_protein}: {pdb_path} with TM-score {row['tm_score']}, Grantham score: {row['grantham_score']}, rmsd: {row['RMSD']}")
    print(f"Sequence identity: {row['pident']}%")
    print("\n")
    pymol_script += f"load {pdb_path}\n"
    pymol_script += f"load /Users/pqh443/Documents/Git_projects/WP2_SweetProteinDiscovery/1_viridiplantae_prot_db/4_sweet_proteins/sweet_protein_pdbs/{row['query']}.pdb\n"        
    pymol_script += f"align {row['query']}, {pdb_name}\n"
    pymol_script += f"color grey, {row['query']}\n"
    pymol_script += f"color green, {pdb_name}\n"
    with open(os.path.join(pymol_script_path, f"{sweet_protein}_best_hit.pml"), "w") as f:
        f.write(pymol_script)

Best hit for Brazzein_4HE7: /Users/pqh443/Documents/Git_projects/WP2_SweetProteinDiscovery/6_structural_analyses/3_representative_structures/Brazzein_4HE7/pdb_dir/cropped_af3_Brazzein_4HE7___A0ABM0YZR9.pdb with TM-score 0.85409, Grantham score: 78.5, rmsd: nan
Sequence identity: 30.5%


Best hit for Honey_truffle_sweetener: /Users/pqh443/Documents/Git_projects/WP2_SweetProteinDiscovery/6_structural_analyses/3_representative_structures/Honey_truffle_sweetener/pdb_dir/cropped_afdb_Honey_truffle_sweetener___A0A7J8XYF8.pdb with TM-score 0.80694, Grantham score: 65.0, rmsd: 2.4
Sequence identity: 16.0%


Best hit for MNEI_1IV7: /Users/pqh443/Documents/Git_projects/WP2_SweetProteinDiscovery/6_structural_analyses/3_representative_structures/MNEI_1IV7/pdb_dir/cropped_of3_MNEI_1IV7___OneKP_scaffold-QSKP-2003977-Polansia_trachysperma.pdb with TM-score 0.81243, Grantham score: 111.14285714285714, rmsd: nan
Sequence identity: 25.0%


Best hit for Miraculin: /Users/pqh443/Documents/Git_projects/WP2

# Make OpenFold 3 JSON files for complex modelling with the sweet taste receptor heterodimer

In [47]:
msa_dir = "/dcai/projects01/cu_0027/sweet_protein_complexes"
local_msa_dir = "/Users/pqh443/Desktop/msas/"
local_zip_dir = "/Users/pqh443/Desktop/alignments/"
for index, row in representative_hits_df.iterrows():
    sweet_protein_seq = row["cropped_tseq"]
    sweet_protein_name = row["target"]
    if row["source"] == "UniProt":
        sweet_protein_name = row["target"].split('|')[1]
    else:
        sweet_protein_name = row["target"]

    sweet_protein_name = row["query"] + "___" + sweet_protein_name
    sweet_protein_name = sweet_protein_name.replace("+", "_")
    chain_dict = {
        "A": {"seq": receptor_seq_dict["ts1r2_human"], "msa_path": f"{msa_dir}/alignments/ts1r2_human/uniref90_hits.a3m"}, 
        "B": {"seq":  receptor_seq_dict["ts1r3_human"], "msa_path": f"{msa_dir}/alignments/ts1r3_human/uniref90_hits.a3m"},
        "C": {"seq": sweet_protein_seq, "msa_path": f"{msa_dir}/alignments/{sweet_protein_name}/uniref90_hits.a3m"}    
    }

    # Check that all MSA files exist
    for chain in chain_dict.keys():
        msa_path = chain_dict[chain]["msa_path"]
        if chain == "A":
            local_path = os.path.join(local_msa_dir, "receptor", "ts1r2_human.a3m")
        elif chain == "B":
            local_path = os.path.join(local_msa_dir, "receptor", "ts1r3_human.a3m")
        elif chain == "C":
            local_path = os.path.join(local_msa_dir, "peptide", f"{sweet_protein_name}.a3m")
        else:   
            print(f"Unknown chain {chain} for {sweet_protein_name}. Skipping MSA check for this chain.")
            continue

        if not os.path.exists(local_path):
            print(f"Warning: MSA file {local_path} does not exist for chain {chain} of {sweet_protein_name}. Skipping this entry.")
            continue
        else:
            # Copy MSA file to expected location if it doesn't exist there
            if chain == "A":
                zip_subdir = os.path.join(local_zip_dir, "ts1r2_human")
            elif chain == "B":
                zip_subdir = os.path.join(local_zip_dir, "ts1r3_human")
            elif chain == "C":
                zip_subdir = os.path.join(local_zip_dir, sweet_protein_name)
                
            # Check if the zipped MSA file exists
            zipped_msa_path = os.path.join(zip_subdir, "uniref90_hits.a3m")
            if not os.path.exists(zipped_msa_path):
                os.makedirs(zip_subdir, exist_ok=True)
                os.system(f"cp {local_path} {zipped_msa_path}")

    of3_json(chain_dict, model_name=sweet_protein_name, num_recycles=3, json_path=f"{repo_dir}/6_structural_analyses/3_representative_structures/of3_jsons/{sweet_protein_name}.json", use_paired_msas=False)

# Make the same JSON files for the known sweet tasting proteins

In [48]:
sweet_protein_dict = {}
for sweet_protein in representative_hits_df["query"].unique():
    # Read sweet protein sequence from fasta file
    fasta_file = os.path.join(fasta_dir, f"{sweet_protein}.fasta")
    fasta_dict = fasta_to_dict(fasta_file)
    sweet_protein_seq = list(fasta_dict.values())[0]
    sweet_protein_dict[sweet_protein] = sweet_protein_seq

for key, item in sweet_protein_dict.items():
    sweet_protein_seq = item
    sweet_protein_name = key
    chain_dict = {
        "A": {"seq": receptor_seq_dict["ts1r2_human"], "msa_path": f"{msa_dir}/alignments/ts1r2_human/uniref90_hits.a3m"}, 
        "B": {"seq":  receptor_seq_dict["ts1r3_human"], "msa_path": f"{msa_dir}/alignments/ts1r3_human/uniref90_hits.a3m"},
        "C": {"seq": sweet_protein_seq, "msa_path": f"{msa_dir}/alignments/{sweet_protein_name}/uniref90_hits.a3m"}    
    }

    # Check that all MSA files exist
    for chain in chain_dict.keys():
        msa_path = chain_dict[chain]["msa_path"]
        if chain == "A":
            local_path = os.path.join(local_msa_dir, "receptor", "ts1r2_human.a3m")
        elif chain == "B":
            local_path = os.path.join(local_msa_dir, "receptor", "ts1r3_human.a3m")
        elif chain == "C":
            local_path = os.path.join(local_msa_dir, "peptide", f"{sweet_protein_name}.a3m")
        else:   
            print(f"Unknown chain {chain} for {sweet_protein_name}. Skipping MSA check for this chain.")
            continue

        if not os.path.exists(local_path):
            print(f"Warning: MSA file {local_path} does not exist for chain {chain} of {sweet_protein_name}. Skipping this entry.")
            continue
        else:
            # Copy MSA file to expected location if it doesn't exist there
            if chain == "A":
                zip_subdir = os.path.join(local_zip_dir, "ts1r2_human")
            elif chain == "B":
                zip_subdir = os.path.join(local_zip_dir, "ts1r3_human")
            elif chain == "C":
                zip_subdir = os.path.join(local_zip_dir, sweet_protein_name)
                
            # Check if the zipped MSA file exists
            zipped_msa_path = os.path.join(zip_subdir, "uniref90_hits.a3m")
            if not os.path.exists(zipped_msa_path):
                os.makedirs(zip_subdir, exist_ok=True)
                os.system(f"cp {local_path} {zipped_msa_path}")

    of3_json(chain_dict, model_name=sweet_protein_name, num_recycles=3, json_path=f"{repo_dir}/6_structural_analyses/3_representative_structures/of3_jsons/{sweet_protein_name}.json", use_paired_msas=False)
